In [ ]:
# Building a binary classifier model with PyTorch

This notebook address following steps:
- Build a `PyTorch` `Dataset`/`DataLoader` pipeline
- Build a small fully-connected neural network with `PyTorch`
- Build a training loop (with validation) with `PyTorch`
- Draw plots: training/validation loss, accuracy, ROC curve, and confusion matrix
- Save and load a model


In [3]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, accuracy_score, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm


# Seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)


# Data import, split, transform
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1).values
y = df['target'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state = RANDOM_STATE)
sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_val = sc.transform(X_val)


# Dataset Dataloader
class CancerDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = CancerDataset(X_train, y_train)
val_dataset = CancerDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)


# model
class BCNet(nn.Module):
    def __init__(self, input_dim, hidden1=30, hidden2=15, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.out = nn.Linear(hidden2, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

nb_input_features = len(train_dataset[0][0])


In [ ]:
# training

writer = SummaryWriter(log_dir="./My_first_NN")         # TensorBoard writer instanciation. Writer will output to ./runs/ directory by default

model = BCNet(nb_input_features)
criterion = nn.BCEWithLogitsLoss()  # BCEWithLogitsLoss by default returns average of losses given a batch...
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in tqdm(range(1, 50)):
    model.train()
    running_loss = 0
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(x_batch)# ...So we need to multiply this value by the number of samples in the batch

    train_loss = running_loss / len(train_loader.dataset) # Divide by the total number of samples
    writer.add_scalar("Loss/train", train_loss, epoch)    # log to writer

writer.close()

100%|██████████| 49/49 [00:00<00:00, 59.48it/s]


In [6]:
# On Jupyter Notebbook or vscode:
# tensorboard --logdir=runs  # then go to --> http://localhost:6006

# on Google colab:
%load_ext tensorboard
%tensorboard --logdir "./My_first_NN"